# Powerful RAG with Langchain Contextual Compression Retriever

In [1]:
%pip install -q langchain langchain-community langchain-core langchain-groq langchain-huggingface chromadb sentence-transformers pypdf python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install langchain_groq

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Get the current working directory
current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")

# Try to find .env file in current directory and parent directories
env_path = Path(".env")
if not env_path.exists():
    # Try parent directory
    env_path = Path("../.env")
    if not env_path.exists():
        # Try the GenAI-Notes root
        env_path = Path("../../.env")

print(f"Looking for .env at: {env_path.absolute()}")

# Load the .env file
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f".env file found and loaded from: {env_path.absolute()}")
else:
    load_dotenv()
    print("No .env file found, checking environment variables...")

# Get the Groq API key
groq_key = os.getenv("GROQ_API_KEY")

if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key
    print("✓ GROQ_API_KEY loaded successfully")
else:
    print("✗ GROQ_API_KEY not found in .env file or environment variables")
    print("Please ensure GROQ_API_KEY is set in your .env file")

Current working directory: d:\GenAI-Notes\AdvanceRAG
Looking for .env at: d:\GenAI-Notes\AdvanceRAG\.env
.env file found and loaded from: d:\GenAI-Notes\AdvanceRAG\.env
✓ GROQ_API_KEY loaded successfully


In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

C:\Users\LOQ\AppData\Local\Temp\ipykernel_11336\4204570146.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

print("Groq model initialized successfully")

Groq model initialized successfully


In [11]:
text_splitter = CharacterTextSplitter(chunk_size=800, chunk_overlap=100)

In [12]:
loader = TextLoader("india_gdp_information.txt")
documents = loader.load()

In [13]:
texts = text_splitter.split_documents(documents)

In [16]:
import os
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = text_splitter.split_documents(documents)

print(f"Created {len(documents)} chunks")

Created 3 chunks


In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully")

Embeddings model loaded successfully


In [21]:
import textwrap
# Helper function for printing docs
def pretty_print_docs(docs):
  print(
      f"\n{'-' * 100}\n".join(
      [
          f"Document {i+1}:\n\n{textwrap.fill(d.page_content, width=80)}\nMetadata: {d.metadata}"
          for i, d in enumerate(docs)
      ]
  )
)

In [22]:
pretty_print_docs(docs)

Document 1:

India’s GDP Overview  India has one of the largest economies in the world and is
considered one of the fastest-growing major economies. Gross Domestic Product
(GDP) represents the total monetary value of all goods and services produced
within the country during a specific period.  Key Facts About India’s GDP  -
India is among the top 5 largest economies globally by nominal GDP. -   The
economy is driven by sectors such as services, manufacturing,     agriculture,
information technology, and telecommunications. -   The services sector
contributes the largest share to India’s GDP. -   Major industries include IT
services, pharmaceuticals, textiles,     automobiles, and financial services. -
India has experienced significant economic reforms since 1991, which
accelerated growth and globalization.  GDP Measurement Types  1.  Nominal GDP
-   Calculated using current market prices.     -   Used for international
economic comparisons. 2.  Real GDP     -   Adjusted for inflation. 

In [23]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever()

print("Vector store created successfully")

Vector store created successfully


In [24]:
# Now that the retriever exists, you can invoke it.
docs = retriever.invoke("Which are the top 3 places to visit in Mauritius?")

In [25]:
docs3 = retriever.invoke("What are the top 3 bars to visit in Mauritius?")

In [26]:
pretty_print_docs(docs3)

Document 1:

India’s GDP growth plays an important role in the global economy and influences
international trade and investment trends.
Metadata: {'source': 'india_gdp_information.txt'}
----------------------------------------------------------------------------------------------------
Document 2:

GDP Measurement Types  1.  Nominal GDP     -   Calculated using current market
prices.     -   Used for international economic comparisons. 2.  Real GDP     -
Adjusted for inflation.     -   Reflects actual growth in production and
services. 3.  GDP Per Capita     -   GDP divided by population.     -
Indicates average economic output per person.  Factors Influencing India’s GDP
-   Population growth -   Government policies -   Infrastructure development -
Foreign Direct Investment (FDI) -   Exports and imports -   Technological
advancement -   Agricultural production  Economic Challenges  -   Income
inequality -   Unemployment -   Inflation -   Rural development gaps -
Environmental sustaina

In [28]:
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
query = "What is the document about?"

response = qa_chain.invoke({"query": query})

print(response["result"])